# 🎯 [MOV02_M01] 전장 수송 자산(트럭, 트레일러, 수송기) 제원 및 도로망 네트워크 데이터 스키마 정의
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M01 |
| **미션 주제** | 전장 수송 자산(트럭, 트레일러, 수송기) 제원 및 도로망 네트워크 데이터 스키마 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 전장 수송 자산(트럭, 트레일러, 수송기) 제원 및 도로망 네트워크 데이터 스키마 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전장 수송 자산(트럭, 트레일러) 제원 및 도로망 데이터 스키마를 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **트럭·트레일러·수송기의 교육용 가정 제원과 도로망(노드·간선·교량 하중) 데이터 스키마를 정의하고, 수송지원 실적·비용 보고 업무를 AS-IS/TO-BE로 비교한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **자산(적재량·속도·km당 비용), 노드, 간선(거리·도로 등급·교량 하중·통행 여부), 수송 요청의 필드와 허용값을 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/mov02_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전장 수송 자산(트럭·트레일러·수송기) 제원과 도로망(노드·간선) 데이터 스키마를 정의해 가상 샘플을 검증하고, 수송지원 실적·비용 보고 업무를 AS-IS/TO-BE 단계로 분해해 병목 구간을 도출한다."""
import copy
import heapq
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

# 1) 스키마 정의: 수송 자산 / 도로망 노드 / 도로 구간(간선) / 수송 요청
SCHEMA = {
    "asset": {"asset_id": {"type": str, "pattern": r"^(TRK|TRL|AIR)-\d{3}$"},
              "kind": {"type": str, "enum": list(ASSETS)},
              "cap_t": {"type": (int, float), "min": 0.5, "max": 50},
              "speed_kmh": {"type": (int, float), "min": 5, "max": 900},
              "cost_km": {"type": (int, float), "min": 0},
              "status": {"type": str, "enum": ["가용", "정비", "운행중"]}},
    "node": {"node_id": {"type": str, "pattern": r"^N\d{4}$"},
             "x_km": {"type": (int, float), "min": -100, "max": 100},
             "y_km": {"type": (int, float), "min": -100, "max": 100}},
    "edge": {"id": {"type": str, "pattern": r"^E\d{3}$"},
             "u": {"type": str, "pattern": r"^N\d{4}$"}, "v": {"type": str, "pattern": r"^N\d{4}$"},
             "km": {"type": (int, float), "min": 0.1, "max": 200},
             "road": {"type": str, "enum": list(ROAD_FACTOR)},
             "bridge": {"type": bool}, "open": {"type": bool}},
    "request": {"req_id": {"type": str, "pattern": r"^REQ-\d{4}$"},
                "dest": {"type": str, "pattern": r"^N\d{4}$"},
                "cargo_t": {"type": (int, float), "min": 0.1, "max": 200},
                "deadline_h": {"type": (int, float), "min": 0.5, "max": 72}},
}


def check(name, value, rule):
    ok_type = isinstance(value, rule["type"]) and (rule["type"] is bool or not isinstance(value, bool))
    if not ok_type:
        return [f"{name}: 자료형 오류({type(value).__name__})"]
    errs = []
    if "enum" in rule and value not in rule["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{value}'")
    if "min" in rule and value < rule["min"]:
        errs.append(f"{name}: 최소값 {rule['min']} 미만")
    if "max" in rule and value > rule["max"]:
        errs.append(f"{name}: 최대값 {rule['max']} 초과")
    if "pattern" in rule and not re.match(rule["pattern"], value):
        errs.append(f"{name}: 형식 불일치 '{value}'")
    return errs


def validate(group, rec, node_ids=None):
    errs = []
    for k, rule in SCHEMA[group].items():
        errs += [f"{group}.{k}: 필수 필드 누락"] if k not in rec else check(f"{group}.{k}", rec[k], rule)
    if group == "edge" and node_ids is not None:          # 참조 무결성: 없는 노드를 잇는 간선
        errs += [f"edge.{k}: 존재하지 않는 노드 {rec[k]}" for k in ("u", "v") if rec.get(k) not in node_ids]
    if group == "edge" and rec.get("bridge") and not rec.get("limit_t"):
        errs.append("edge.limit_t: 교량 하중 제한 누락")
    return errs


# 2) [교육용가상자료] 샘플 생성: 자산 12대, 도로망 48노드, 수송 요청 6건
nodes, edges = make_network()
rng = np.random.default_rng(1)
prefix = {"truck": "TRK", "trailer": "TRL", "airlift": "AIR"}
assets = [{"asset_id": f"{prefix[k]}-{i:03d}", "kind": k, **{f: ASSETS[k][f] for f in ("cap_t", "speed_kmh", "cost_km")},
           "status": str(rng.choice(["가용", "정비", "운행중"], p=[0.7, 0.15, 0.15]))}
          for i, k in enumerate(["truck"] * 7 + ["trailer"] * 3 + ["airlift"] * 2)]
node_recs = [{"node_id": n, "x_km": round(x, 2), "y_km": round(y, 2)} for n, (x, y) in nodes.items()]
requests = [{"req_id": f"REQ-{i:04d}", "dest": f"N{r:02d}07", "cargo_t": float(round(rng.uniform(3, 30), 1)),
             "deadline_h": float(rng.choice([2, 3, 4]))} for i, r in enumerate(range(6))]
samples = [("asset", a) for a in assets] + [("node", n) for n in node_recs] + [("edge", e) for e in edges] + [("request", q) for q in requests]
ids = set(nodes)
n_err = sum(1 for g, r in samples if validate(g, r, ids))
print(f"정상 샘플 {len(samples)}건(자산 {len(assets)}, 노드 {len(node_recs)}, 간선 {len(edges)}, 요청 {len(requests)}) 검증 오류 {n_err}건")

# 3) 오류 주입 시험
bad = [("asset", {**assets[0], "asset_id": "TRUCK-1"}), ("asset", {**assets[1], "kind": "tank"}),
       ("edge", {**edges[0], "v": "N9999"}), ("edge", {**{k: v for k, v in edges[1].items() if k != "km"}}),
       ("edge", {**edges[2], "bridge": True, "limit_t": None}), ("request", {**requests[0], "cargo_t": -2.0})]
print("\n[오류 주입 결과]")
caught = 0
for g, r in bad:
    e = validate(g, copy.deepcopy(r), ids)
    caught += bool(e)
    print(" -", "; ".join(e))
print(f"주입 오류 {len(bad)}종 중 {caught}종 검출")

# 4) 수송지원 실적·비용 보고 업무 AS-IS 분해(단계별 소요 분, 가상 측정값)와 TO-BE 개선
steps = [("수송 요청 접수(전화·메신저)", 20, "요청 서식 표준화·자동 등록", 2),
         ("가용 자산 확인(차량 일지 수기 대조)", 30, "자산 상태 스키마로 자동 조회", 2),
         ("도로 상황 확인(구두 보고 취합)", 40, "도로 제약 자동 반영(미션 2)", 5),
         ("경로 선정(지도 수작업)", 35, "최단·위협회피 경로 자동 추천", 3),
         ("배차·적재 계획 작성", 25, "적재 효율 기반 자동 배차", 5),
         ("수송통제관 승인", 15, "HITL 승인 화면(근거 자동 첨부)", 10),
         ("운행 기록 수기 작성", 20, "운행 이벤트 자동 기록", 3),
         ("지연·사고 보고 작성", 25, "지연 원인 코드 선택식 기록", 5),
         ("운행 실적 집계(엑셀 복사·붙여넣기)", 60, "실적 자동 집계", 1),
         ("비용 산출(유류·거리 단가 계산)", 45, "자산별 km 단가 자동 계산", 1),
         ("실적·비용 보고서 작성", 50, "대시보드 화면 캡처 대신 자동 요약", 5),
         ("상급 부대 보고·수정 요청 반영", 30, "단일 데이터 기준으로 수정 최소화", 10)]
proc = pd.DataFrame(steps, columns=["업무 단계", "AS-IS(분)", "TO-BE 개선", "TO-BE(분)"])
proc.index = range(1, len(proc) + 1)
thr = proc["AS-IS(분)"].mean() + 0.5 * proc["AS-IS(분)"].std()
proc["병목"] = np.where(proc["AS-IS(분)"] >= thr, "병목", "")
print(f"\n[AS-IS / TO-BE 업무 단계 {len(proc)}단계] (병목 기준: 평균+0.5표준편차 = {thr:.0f}분 이상)")
print(proc[["업무 단계", "AS-IS(분)", "TO-BE(분)", "병목"]].to_string())
nb = int((proc["병목"] == "병목").sum())
asis, tobe = proc["AS-IS(분)"].sum(), proc["TO-BE(분)"].sum()
print(f"\n업무 단계 {len(proc)}단계(목표 10단계 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {nb}개소(목표 3개소 이상: {'달성' if nb >= 3 else '미달'})")
print(f"1회 처리 시간 AS-IS {asis}분 -> TO-BE {tobe}분 ({1 - tobe / asis:.0%} 단축 예상)")
print("\n[스키마 그룹별 필드 수]", {g: len(f) for g, f in SCHEMA.items()})


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
정상 샘플 157건(자산 12, 노드 48, 간선 91, 요청 6) 검증 오류 0건

[오류 주입 결과]
 - asset.asset_id: 형식 불일치 'TRUCK-1'
 - asset.kind: 허용되지 않은 값 'tank'
 - edge.v: 존재하지 않는 노드 N9999
 - edge.km: 필수 필드 누락
 - edge.limit_t: 교량 하중 제한 누락
 - request.cargo_t: 최소값 0.1 미만
주입 오류 6종 중 6종 검출

[AS-IS / TO-BE 업무 단계 12단계] (병목 기준: 평균+0.5표준편차 = 40분 이상)
                    업무 단계  AS-IS(분)  TO-BE(분)  병목
1        수송 요청 접수(전화·메신저)        20         2    
2   가용 자산 확인(차량 일지 수기 대조)        30         2    
3      도로 상황 확인(구두 보고 취합)        40         5  병목
4           경로 선정(지도 수작업)        35         3    
5             배차·적재 계획 작성        25         5    
6                수송통제관 승인        15        10    
7             운행 기록 수기 작성        20         3    
8             지연·사고 보고 작성        25         5    
9    운행 실적 집계(엑셀 복사·붙여넣기)        60         1  병목
10     비용 산출(유류·거리 단가 계산)        45         1  병목
11           실적·비용 보고서 작성        50         5  병목
12      상급 부대 보고·수정 요청 반영        30        10    

업무 단계 12단계(목표 10단계 이상: 달성), 병목 4개소(목표 3개소 이상: 달성)
1회 처리 시간 AS-IS 395분 -> TO-BE 52분 (87% 단축 예상)

[스키마 그룹별 필드 수] {'asset': 6, 'node': 3, 'edge': 7, 'request': 4}
```
</details>
